# M2E3 - Preprocessing
In this notebook, `fastp` will be utilized on the Hellbender Desktop VNC Client, and the sample organization will be similar to the previous exercise, M2E1. It is essential to follow along using your own VNC connection to familiarize yourself with the use of the Hellbender desktop, as well as how to load environments, run tools, and parameterize them within the terminal. 
**You will be graded on completion of the preprocessing steps within this exercise, so follow along and ask for help on the Week 2 Questions and Help Canvas Discussion if you have any issues.**

This Notebook will cover: 
- Organization of `m2_project_fastp_usage` folder
- Cadmium data from the McInturf Arabidopsis data set ([GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157))
   - Read through the introduction and RNA-Seq Analysis sections of [this paper(McInturf, 2022)](https://doi.org/10.1111/pce.13192)
- The usage of `fastp` on this data set
- Questions for the [Module 2 Discussion on Canvas](https://umsystem.instructure.com/courses/382808/modules/items/11020460).


```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## Organization of the `m2_project_fastp_usage` Folder

The folder `m2_project_fastp_usage` contains the following directories: 
```
m2_project_fastp_usage/
├── data
│   ├── raw → raw reads from roots of col-0 (CRN) and the roots of cd treated col-0 (ORN) fastq files within GSE79275
│   └── trimmed  → trimmed fastq files from the data folder after use of fastp
├── results → where the QC reports will be saved
└── scripts → fastp script used in M2E2
```

Samples are retrieved from [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157). Samples used in this Exercise will be:

| Sample ID | GEO Accession | Condition | Replicate | Description |
|------------|----------------|------------|------------|--------------|
| **CRN1-1** | GSM2090459 | C (untreated) | N-1 | Col-0 untreated root sample |
| **CRN2-1** | GSM2090462 | C (untreated) | N-2 | Col-0 untreated root sample |
| **CRN3-1** | GSM2090465 | C (untreated) | N-3 | Col-0 untreated root sample |
| **ORN1-1** | GSM2090477 | O (CdCl₂-treated) | N-1 | Roots treated with 20 µM CdCl₂ for 72 h |
| **ORN2-1** | GSM2090480 | O (CdCl₂-treated) | N-2 | Roots treated with 20 µM CdCl₂ for 72 h |
| **ORN3-1** | GSM2090483 | O (CdCl₂-treated) | N-3 | Roots treated with 20 µM CdCl₂ for 72 h |


## Follow along in M2E2V9 to run `fastp` on our exapmle data set within the `m2_project_fastp_usage` folder:

<iframe width="560" height="315" src="https://www.youtube.com/embed/-tWwKvrzEac?si=_B7a_GACAdurb_-m" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [ ]:
%%bash
#fastp usage to follow along

# navigate to your Module 2 project folder
cd MyStudentFolder/Module_2/m2_project_fastp_usage/

#observe the file structure and the names of the samples.
tree 

for file in data/raw/*.fastq.gz
do
  base=$(basename "$file" .fastq.gz)
  fastp \
    -i "$file" \
    -o "data/trimmed/${base}_trimmed.fastq.gz" \
    --html "results/${base}_fastp.html" \
    --json "results/${base}_fastp.json" \
    --thread 4 \
    --detect_adapter_for_pe \
    --trim_front1 5 \
    --cut_tail \
    --length_required 30
done


# Fastp Review M2P1: an all-in-one tool

`fastp` can be used to assess quality and trim/filter reads in a single command. This program can be ran on the command line, and will be later looped so that you can apply the tool to multiple files with a simple bash script. The inputs and outputs of fastp are shown below: 

1. Inputs

fastp primarily takes FASTQ files as input. These can be:

- Single-end reads: one FASTQ file per sample.

- Paired-end reads: two FASTQ files per sample (R1 and R2).

Parameters can be added to fastp to adjust trimming with more specificity: 

| Option             | Meaning                                             |
| ------------------ | --------------------------------------------------- |
| `-i`               | Input FASTQ file for R1 (forward reads)             |
| `-I`               | Input FASTQ file for R2 (reverse reads, paired-end) |
| `-o`               | Output trimmed FASTQ for R1                         |
| `-O`               | Output trimmed FASTQ for R2                         |
| `-h`               | Generate an HTML QC report                          |
| `-j`               | Generate a JSON QC report                           |
| `-q 20`            | Trim bases with quality < 20                        |
| `-l 20`            | Discard reads shorter than 20 bp after trimming     |
| `--n_base_limit 5` | Discard reads with >5 ambiguous bases               |
| `-w 4`             | Use 4 CPU threads for faster processing             |


2. Outputs:

After running fastp, you get:

- Trimmed FASTQ files (mandatory)

   - Single-end: one output FASTQ file (-o).

  - Paired-end: two output FASTQ files (-o and -O) for R1 and R2.

- QC Reports (optional but **HIGHLY recommended**)

    - HTML report (-h) → interactive summary with plots.

    - JSON report (-j) → structured data for programmatic use.

Note: Filtered reads are removed by trimming/quality/length/N-base filters and are excluded from output FASTQ files.

Below is an example of a fastp command: 

```
fastp \
  -i sample_R1.fastq \
  -I sample_R2.fastq \
  -o sample_R1_trimmed.fastq \
  -O sample_R2_trimmed.fastq \
  -h sample_fastp_report.html \
  -j sample_fastp_report.json \
  -q 20 \
  -l 20 \
  --n_base_limit 5 \
  -w 4
```

# Discussion: Comparing QC results before and after trimming

**Post on the Module 2 Discussion the answers to the following questions:**

-    How did the per-base quality scores change after trimming?
-   Did trimming improve the average quality near the ends of the reads?
-   Look at the sequence length distribution plots:
     -   What happened to the read lengths after trimming, and why?
-   Did the number of reads failing QC decrease after trimming?
- Why is it important to perform quality control both before and after trimming?
-  What might happen if we skip trimming or don’t assess quality afterward?

